In [1]:
import sys
import os
parent_dir = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.append(parent_dir)
import pandas as pd
import numpy as np
from src.preprocessing import scale_features, add_outlier_flags, target_feature_split, one_hot_encoding
from src.feature_engineering import (create_vocal_instrumental_ratio, create_energy_rhythm_interaction, create_moodscore_bins
                                     ,create_vocal_energy_ratio,create_energy_acoustic_ratio,create_mood_rhythm_interaction,create_live_track_interaction)
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, StackingRegressor
from lightgbm import LGBMRegressor
from sklearn.dummy import DummyRegressor
from sklearn.metrics import root_mean_squared_error, mean_absolute_error
from sklearn.model_selection import KFold, cross_val_score
from sklearn.inspection import permutation_importance
#import featuretools as ft
from sklearn.preprocessing import MinMaxScaler, PolynomialFeatures
from autofeat import AutoFeatRegressor
import matplotlib.pyplot as plt


w:\Git\Estudos\Predicting-the-Beats-per-Minute-of-Songs-Kaggle\.venv\Lib\site-packages\woodwork\__init__.py:2: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


2025-09-23 18:35:19,230 featuretools - WARNING    While loading primitives via "premium_primitives" entry point, ignored primitive "DiversityScore" from "premium_primitives.diversity_score" because a primitive with that name already exists in "nlp_primitives.diversity_score"
2025-09-23 18:35:19,231 featuretools - WARNING    While loading primitives via "premium_primitives" entry point, ignored primitive "LSA" from "premium_primitives.lsa" because a primitive with that name already exists in "nlp_primitives.lsa"
2025-09-23 18:35:19,233 featuretools - WARNING    While loading primitives via "premium_primitives" entry point, ignored primitive "MeanCharactersPerSentence" from "premium_primitives.mean_characters_per_sentence" because a primitive with that name already exists in "nlp_primitives.mean_characters_per_sentence"
2025-09-23 18:35:19,234 featuretools - WARNING    While loading primitives via "premium_primitives" entry point, ignored primitive "NumberOfSentences" from "premium_primi

In [2]:
# Input Data
df_train = pd.read_csv("../data/raw/train.csv")
df_test = pd.read_csv("../data/raw/test.csv")
df_sample_submission = pd.read_csv("../data/raw/sample_submission.csv")

In [3]:
flag_autofeat = False

In [4]:
if flag_autofeat:
    X = df_train.drop(columns=["BeatsPerMinute", "id"])
    y = df_train["BeatsPerMinute"]

    X_sample = X.sample(150000, random_state=42)
    y_sample = y.loc[X_sample.index]

    afreg = AutoFeatRegressor(
        verbose=1, 
        featsel_runs=1 
    )

    #X_auto = afreg.fit_transform(X, y)
    X_auto = afreg.fit_transform(X_sample, y_sample)

    print("Original Shape:", X_sample.shape)
    print("Expanded Shape:", X_auto.shape)
    print(X_auto.head())

In [5]:
#preprocessing
df_train, scaler = scale_features(df_train, ['AudioLoudness','TrackDurationMs'])
df_train, _ = add_outlier_flags(df_train, ['RhythmScore','AudioLoudness','VocalContent','AcousticQuality','InstrumentalScore','LivePerformanceLikelihood','TrackDurationMs'])

df_test, _ = scale_features(df_test,  ['AudioLoudness','TrackDurationMs'], scaler)
df_test, _ = add_outlier_flags(df_test, ['RhythmScore','AudioLoudness','VocalContent','AcousticQuality','InstrumentalScore','LivePerformanceLikelihood','TrackDurationMs'])

In [6]:
# feature engineering
df_train = create_vocal_instrumental_ratio(df_train)
df_train = create_energy_rhythm_interaction(df_train)
df_train = create_moodscore_bins(df_train)
df_train = create_vocal_energy_ratio(df_train)
df_train = create_energy_acoustic_ratio(df_train)
df_train = create_mood_rhythm_interaction(df_train)
df_train = create_live_track_interaction(df_train)


df_test = create_vocal_instrumental_ratio(df_test)
df_test = create_energy_rhythm_interaction(df_test)
df_test = create_moodscore_bins(df_test)
df_test = create_vocal_energy_ratio(df_test)
df_test = create_energy_acoustic_ratio(df_test)
df_test = create_mood_rhythm_interaction(df_test)
df_test = create_live_track_interaction(df_test)

In [7]:
#also preprocessing
df_train, encoder = one_hot_encoding(df_train, 'MoodScore_bins')
df_test, _ = one_hot_encoding(df_test, 'MoodScore_bins', encoder)

In [8]:
#Definition of X and y
X_train, y_train = target_feature_split(df=df_train, target="BeatsPerMinute", exclude_cols=['id'])
X_test, y_test = target_feature_split(df=df_test, exclude_cols=['id'])

In [9]:
RANDOM_STATE = 42
kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

In [10]:
models = {
    "dummy": DummyRegressor(strategy="mean"),
    "ridge": Ridge(random_state=RANDOM_STATE),
    "rf": RandomForestRegressor(max_depth=12,
                                min_samples_leaf=5, n_jobs=-1,
                                random_state=RANDOM_STATE),
    "lgbm": LGBMRegressor(n_jobs=-1, random_state=RANDOM_STATE)
}

oof = {name: np.zeros(len(X_train)) for name in models.keys()}

In [11]:
for i, (train_index, val_index) in enumerate(kf.split(X_train, y_train)):
    print(f"Fold {i}:")
    print(f"  Train: index={train_index}")
    print(f"  Validation:  index={val_index}")
    X_tr, X_val = X_train.iloc[train_index], X_train.iloc[val_index]
    y_tr, y_val = y_train.iloc[train_index], y_train.iloc[val_index]

    print(f"Dummy:")
    models['dummy'].fit(X_tr, y_tr)
    oof['dummy'][val_index] = models['dummy'].predict(X_val)

    print(f"Ridge:")
    models['ridge'].fit(X_tr, y_tr)
    oof['ridge'][val_index] = models['ridge'].predict(X_val)

    print(f"Random Forest:")
    models['rf'].fit(X_tr, y_tr)
    oof['rf'][val_index] = models['rf'].predict(X_val)

    print(f"LGBM:")
    models['lgbm'].fit(X_tr, y_tr)
    oof['lgbm'][val_index] = models['lgbm'].predict(X_val)

Fold 0:
  Train: index=[     0      1      3 ... 524159 524160 524162]
  Validation:  index=[     2      6      7 ... 524146 524161 524163]
Dummy:
Ridge:
Random Forest:
LGBM:
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.054933 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3841
[LightGBM] [Info] Number of data points in the train set: 419331, number of used features: 23
[LightGBM] [Info] Start training from score 119.056554
Fold 1:
  Train: index=[     1      2      3 ... 524160 524161 524163]
  Validation:  index=[     0     11     16 ... 524153 524159 524162]
Dummy:
Ridge:
Random Forest:
LGBM:
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.041316 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Inf

In [12]:
#baseline rmse's and mae's
#dummy RMSE: 26.468078417587844 MAE: 21.19987341149104
#ridge RMSE: 26.466341374596727 MAE: 21.19792095039579
#rf RMSE: 26.465788481708348 MAE: 21.197653869516085
#lgbm RMSE: 26.467350811600735 MAE: 21.198498087056862

In [13]:
rmses = {}
for name, preds in oof.items():
    print(name, "RMSE:", root_mean_squared_error(y_train, preds), "MAE:", mean_absolute_error(y_train, preds))
    rmses

dummy RMSE: 26.468078417587844 MAE: 21.19987341149104
ridge RMSE: 26.465385825000094 MAE: 21.197030571261287
rf RMSE: 26.465305023507657 MAE: 21.196963534692447
lgbm RMSE: 26.467526741672962 MAE: 21.198457247155215


In [14]:
dict_importances = {}
for model in models.keys():
    print(model)
    perm = permutation_importance(models[model], X_train, y_train, scoring="neg_root_mean_squared_error", n_repeats=10, random_state=RANDOM_STATE, n_jobs=-1)
    importance_df = pd.DataFrame({"feature": X_train.columns, "perm_importance": perm["importances_mean"]})
    importance_df = importance_df.sort_values("perm_importance", ascending=False)
    dict_importances[model] = importance_df

dummy
ridge
rf
lgbm


In [15]:
dict_importances['dummy']


,feature,perm_importance
0,RhythmScore,0.0
1,AudioLoudness,0.0
22,MoodScore_bins_low,0.0
21,live_track_interaction,0.0
20,mood_rhythm_interaction,0.0
19,energy_acoustic_ratio,0.0
18,vocal_energy_ratio,0.0
17,energy_rhythm_interaction,0.0
16,vocal_instrumental_ratio,0.0
15,is_outlier_TrackDurationMs,0.0


In [16]:
dict_importances['ridge']


,feature,perm_importance
20,mood_rhythm_interaction,6.271017e-03
22,MoodScore_bins_low,5.015520e-03
21,live_track_interaction,2.394461e-03
23,MoodScore_bins_medium,1.903113e-03
2,VocalContent,8.633262e-04
18,vocal_energy_ratio,7.251435e-04
6,MoodScore,6.856309e-04
8,Energy,3.016616e-04
19,energy_acoustic_ratio,2.177515e-04
1,AudioLoudness,2.101556e-04


In [17]:
dict_importances['rf']


,feature,perm_importance
6,MoodScore,1.205377e-01
7,TrackDurationMs,8.661553e-02
21,live_track_interaction,8.154580e-02
0,RhythmScore,6.589186e-02
20,mood_rhythm_interaction,5.610121e-02
19,energy_acoustic_ratio,5.420143e-02
2,VocalContent,5.118194e-02
1,AudioLoudness,4.689274e-02
5,LivePerformanceLikelihood,4.539696e-02
18,vocal_energy_ratio,4.193798e-02


In [18]:
dict_importances['lgbm']

,feature,perm_importance
6,MoodScore,0.056618
2,VocalContent,0.055616
7,TrackDurationMs,0.038052
21,live_track_interaction,0.035281
0,RhythmScore,0.034525
18,vocal_energy_ratio,0.033442
5,LivePerformanceLikelihood,0.033361
20,mood_rhythm_interaction,0.032619
17,energy_rhythm_interaction,0.032559
19,energy_acoustic_ratio,0.031098


In [ ]:
stack = StackingRegressor(
    estimators=[
        ("ridge", models['ridge']),
        ("rf", models['rf']),
        ("lgbm", models['lgbm'])
    ],
    final_estimator=Ridge(),
    passthrough=True,
    n_jobs=-1
)

scores = cross_val_score(stack, X_train, y_train, cv=kf, scoring="neg_root_mean_squared_error")
print("Stacking RMSE:", -np.mean(scores))


In [ ]:
#Random Forest submission
y_test_pred = models['rf'].predict(X_test)
df_sample_submission['BeatsPerMinute'] = y_test_pred
df_sample_submission.to_csv("../results/first_random_forest_submission.csv", index=False)

In [ ]:
#Random Forest submission
y_test_pred = stack.predict(X_test)
df_sample_submission['BeatsPerMinute'] = y_test_pred
df_sample_submission.to_csv("../results/first_stack_submission.csv", index=False)